# 04 — Customer Feature Engineering

Build monthly customer snapshots using a **180-day lookback** and a **90-day forward label horizon**. Features must use only information available before the snapshot date. A customer is labelled positive if they make at least one valid purchase in the next 90 days.

The reusable implementation is in `src/features.py`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "features.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import DB_FILE
from src.features import (
    build_customer_snapshots, FEATURES_FILE, LOOKBACK_DAYS,
    HORIZON_DAYS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET
)
import duckdb

print("Lookback:", LOOKBACK_DAYS, "days | Label horizon:", HORIZON_DAYS, "days")

## 1. Load cleaned transactions

In [ ]:
if not DB_FILE.exists():
    raise FileNotFoundError("Run Notebook 02 (or python -m src.data) first.")
with duckdb.connect(str(DB_FILE), read_only=True) as con:
    transactions = con.execute("SELECT * FROM transactions").fetchdf()
print("Transaction rows:", len(transactions))

## 2. Build snapshots

In [ ]:
snapshots = build_customer_snapshots(transactions)
print("Snapshot rows:", len(snapshots))
print("Snapshot dates:", snapshots.snapshot_date.min(), "to", snapshots.snapshot_date.max())
display(snapshots.head())

## 3. Inspect target balance and feature distributions

In [ ]:
display(snapshots.groupby("snapshot_date")[TARGET].agg(["count", "mean"]).rename(columns={"mean":"repeat_purchase_rate"}))
display(snapshots[NUMERIC_FEATURES].describe().T)
display(snapshots[TARGET].value_counts(normalize=True).rename("share").to_frame())

## 4. Persist the modelling table

In [ ]:
FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
snapshots.to_parquet(FEATURES_FILE, index=False)
print("Saved:", FEATURES_FILE)

## Leakage and target checks

- Confirm every feature is derived from dates strictly before its snapshot.
- Confirm the target uses purchases from the snapshot date up to (but not including) snapshot date + 90 days.
- Customers with no purchase in the lookback are not represented in this dataset; the model estimates repeat purchase among customers with recent historical activity.
- Inspect class balance over time before modelling.